<div align="center">

# Day 02: Value-Based Methods

**Reinforcement Learning and Language Model Alignment (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-02) · [Lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html) · [Interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lab.html) · [PDF notes](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/Day02_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Nothing needs to be installed.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this notebook is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
from itertools import product
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.4), 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 9})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok


print('environment ready')

## Python step 2: NumPy arrays and tables of values

Tabular reinforcement learning keeps one estimate per state in a table, which is naturally a NumPy array &#91;[6](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#ref-6)&#93;. This step builds a table with one row per method and one column per state, writes the temporal difference update by hand and estimates the values of a three-state chain with Monte Carlo and temporal difference learning.

### A table of values as an array

In [ ]:
import numpy as np
V = np.zeros((2, 3))               # row 0 Monte Carlo, row 1 temporal difference
V[1, 2] = 0.5
print(V.shape)
print(V)
print("column of state 2:", V[:, 2])
print("largest value in row 1:", V[1].max(), " in state", int(V[1].argmax()))

`np.zeros((2, 3))` creates a table of two rows and three columns: one row per method and one column per state. `V[1, 2]` addresses row 1, column 2, and `V[1]` the whole row of temporal difference learning. In `V[:, 2]` the colon selects every row, so the result is the column of state 2. `argmax` returns the position of the largest value in a row, here the state with the highest estimate, and `max` the value itself.

### One temporal difference update

In [ ]:
alpha, gamma = 0.5, 0.9
s, r, s2 = 1, 0.0, 2               # from state 1, reward 0, next state 2
target = r + gamma * V[1, s2]
V[1, s] += alpha * (target - V[1, s])
print("target:", target, " new V[1, 1]:", V[1, s])

The target is the reward plus the discounted estimate of the next state, as in the temporal difference update of the lecture. The update moves the old estimate a fraction `alpha` of the way towards the target. State 1 learns from the estimate 0.5 of state 2: The target is 0.45, and the estimate of state 1 rises from 0 to 0.225.

### An environment as a function

In [ ]:
def chain_step(s, a):
    """Three states in a row; moving right from state 2 pays 1 and ends the episode."""
    if a == 1 and s == 2:
        return s, 1.0, True
    s2 = min(s + 1, 2) if a == 1 else max(s - 1, 0)
    return s2, 0.0, False

print(chain_step(0, 1), chain_step(2, 1), chain_step(1, 0))

The function returns three values at once, a tuple: The next state, the reward and whether the episode has ended. `min` and `max` keep the state inside the chain.

### Monte Carlo and temporal differences on the chain

In [ ]:
rng = np.random.default_rng(0)
V = np.zeros((2, 3))
alpha = 0.1
for episode in range(500):
    s, done, steps = 0, False, []
    while not done:
        a = 1 if rng.random() < 0.8 else 0        # right with probability 0.8
        s2, r, done = chain_step(s, a)
        target = r if done else r + gamma * V[1, s2]
        V[1, s] += alpha * (target - V[1, s])     # temporal difference: every step
        steps.append((s, r))
        s = s2
    G = 0.0
    for state, reward in reversed(steps):         # Monte Carlo: after the episode
        G = reward + gamma * G
        V[0, state] += alpha * (G - V[0, state])
print(V.round(3))

The `while` loop runs until the episode ends. The agent moves right with probability 0.8 and left otherwise, so an episode can step back before it reaches the reward. The step size is now 0.1. Row 1 receives a temporal difference update after every step. Row 0 waits for the end of the episode, and the list `steps` stores the state and the reward of every step until then. The loop over `reversed(steps)` then builds the return `G` of each visited state backwards and moves the estimate of that state towards it. The Bellman equation of this policy gives the exact values 0.71, 0.81 and 0.95, rounded, and both rows come close to them. Neither row is exact, because a constant step size keeps reacting to the latest episodes.

### The table, readable

In [ ]:
import pandas as pd
table = pd.DataFrame(V.round(3), index=["Monte Carlo", "temporal difference"],
                     columns=["state 0", "state 1", "state 2"])
print(table)
print("state with the largest estimate:", [int(row.argmax()) for row in V])

A pandas data frame adds names to the rows and columns of the array &#91;[7](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#ref-7)&#93;. Looping over a two-dimensional array visits its rows, so the list comprehension reads, for each method, the state with the largest estimate.

**Quick check.** Why is the target `r` alone, without the value of the next state, when the episode ends?

<details><summary>Answer</summary>

Because nothing follows the end of an episode: The value of a terminal state is zero, so only the final reward counts.

</details>

### Exercises for Python step 2

**Exercise 2.1.** Compute the temporal difference target for a reward of 0, a discount of 0.9 and an estimate of 0.81 for the next state, and store it in `target_21`.

In [ ]:
target_21 = None   # your answer

check("Exercise 2.1", target_21, 0.729)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
target_21 = 0 + 0.9 * 0.81

check("Exercise 2.1", target_21, 0.729)

**Exercise 2.2.** Store the temporal difference estimate of state 0 from the table `V`, rounded to two decimals, in `v_td0`.

In [ ]:
v_td0 = None   # your answer

check("Exercise 2.2", v_td0, round(float(V[1, 0]), 2))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
v_td0 = round(float(V[1, 0]), 2)

check("Exercise 2.2", v_td0, round(float(V[1, 0]), 2))

**Exercise 2.3.** In how many states is the Monte Carlo estimate larger than the temporal difference estimate? Store the count in `n_higher`.

In [ ]:
n_higher = None   # your answer

check("Exercise 2.3", n_higher, int((V[0] > V[1]).sum()))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_higher = int((V[0] > V[1]).sum())

check("Exercise 2.3", n_higher, int((V[0] > V[1]).sum()))

## 1. TokenWorld, the Windy Cliff and the scoreboard of Day 1

*Lecture: [Learning without a model](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#learning-without-a-model)*

**From the formulas to the code.** `S` holds the 161 states of TokenWorld and `NONTERM` the 40 prefixes in which a token is still to be chosen. `score_sequence` is the reward table of Day 1, and `step(s, a)` appends a token and returns the next state, the reward and whether the episode has ended. `V_STAR` and `Q_STAR` are the optimal values $V^*(s)$ and $Q^*(s, a)$ of Day 1, computed by value iteration with $\gamma = 1$. Section 2 takes the best token of every prefix from `Q_STAR` for the policy that it evaluates. From section 3 on, both serve as the scoreboard of the learned tables, and no learning method ever sees them.

In [ ]:
# TokenWorld: sequence generation as a Markov decision process.
#
#   state     s_t = the tokens generated so far, the prefix
#   action    a_t = the next token, from the vocabulary
#   transition      s_{t+1} = s_t followed by a_t, deterministic
#   reward    r_t = 0 until the sequence ends, then a programmatic score
#   done      when <eos> is emitted or the length cap is reached
#
# The tiny version below has four tokens and four positions, so the whole MDP can be
# enumerated and solved exactly.

TINY_VOCAB = ["the", "cat", "sat", "<eos>"]     # 4 tokens
EOS = 3
MAXLEN = 4

def score_sequence(seq, vocab=TINY_VOCAB):
    """The true reward: hand-written, deterministic, known to the designer and hidden from the agent.

    A well-formed utterance in this toy language: starts with 'the', contains 'cat' exactly
    once, ends with '<eos>', and is short. Everything else is penalised.
    """
    toks = [vocab[i] for i in seq if vocab[i] != "<eos>"]
    r = 0.0
    if not toks:
        return -1.0                                    # empty utterance
    if toks[0] == "the":     r += 1.0
    else:                    r -= 0.5
    n_cat = toks.count("cat")
    if n_cat == 1:           r += 1.5
    elif n_cat == 0:         r -= 0.5
    else:                    r -= 0.8 * (n_cat - 1)     # repetition is penalised
    if "sat" in toks and "cat" in toks and toks.index("cat") < toks.index("sat"):
        r += 1.0                                       # word order matters
    r -= 0.25 * len(toks)                              # brevity bonus
    return float(r)


def enumerate_states(vocab_n=4, maxlen=MAXLEN):
    """Every reachable prefix. A prefix ending in EOS is terminal."""
    states = [()]
    frontier = [()]
    for _ in range(maxlen):
        nxt = []
        for s in frontier:
            if s and s[-1] == EOS:
                continue
            for a in range(vocab_n):
                nxt.append(s + (a,))
        states += nxt
        frontier = nxt
    return states


def is_terminal(s, maxlen=MAXLEN):
    return len(s) >= maxlen or (len(s) > 0 and s[-1] == EOS)


def step(s, a, maxlen=MAXLEN, vocab=TINY_VOCAB):
    """One transition. The reward is zero until the episode ends."""
    s2 = s + (a,)
    done = is_terminal(s2, maxlen)
    r = score_sequence(s2, vocab) if done else 0.0
    return s2, r, done


def rollout(policy_fn, rng, maxlen=MAXLEN, vocab_n=4):
    """Run one episode. `policy_fn(state) -> action probabilities`."""
    s, traj = (), []
    while not is_terminal(s, maxlen):
        p = policy_fn(s)
        a = int(rng.choice(len(p), p=p))
        s2, r, done = step(s, a, maxlen)
        traj.append((s, a, r))
        s = s2
    return traj, s

In [ ]:
# Windy Cliff: a 4 x 8 gridworld with stochastic transitions.
#
# With probability `slip` the wind pushes the agent sideways. Stepping into the cliff
# costs 100 and returns the agent to the start; every other step costs 1.

ROWS, COLS = 4, 8
START, GOAL = (3, 0), (3, 7)
CLIFF = [(3, c) for c in range(1, 7)]
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]     # up, down, left, right
ANAMES = ["up", "down", "left", "right"]

def grid_step(s, a, slip=0.15, rng=None):
    """With probability `slip` the wind pushes the agent sideways instead."""
    if rng is not None and rng.random() < slip:
        a = int(rng.choice([2, 3]))
    dr, dc = ACTIONS[a]
    r, c = s[0] + dr, s[1] + dc
    r = min(max(r, 0), ROWS - 1); c = min(max(c, 0), COLS - 1)
    s2 = (r, c)
    if s2 in CLIFF:
        return START, -100.0, False       # fall off; back to the start, episode continues
    if s2 == GOAL:
        return s2, 0.0, True
    return s2, -1.0, False

def grid_transitions(slip=0.15):
    """Exact P(s'|s,a) and R(s,a,s') for dynamic programming."""
    P = {}
    for r in range(ROWS):
        for c in range(COLS):
            s = (r, c)
            if s == GOAL:
                continue
            for a in range(4):
                outcomes = {}
                for real_a, prob in [(a, 1 - slip), (2, slip / 2), (3, slip / 2)]:
                    dr, dc = ACTIONS[real_a]
                    rr = min(max(r + dr, 0), ROWS - 1); cc = min(max(c + dc, 0), COLS - 1)
                    s2 = (rr, cc)
                    if s2 in CLIFF:
                        s2, rew = START, -100.0
                    elif s2 == GOAL:
                        rew = 0.0
                    else:
                        rew = -1.0
                    key = (s2, rew)
                    outcomes[key] = outcomes.get(key, 0.0) + prob
                P[(s, a)] = [(p, s2, rew) for (s2, rew), p in outcomes.items()]
    return P

In [ ]:
S = enumerate_states()
NONTERM = [s for s in S if not is_terminal(s)]
NA = len(TINY_VOCAB)

def value_iteration_tokenworld():
    V = {s: 0.0 for s in S}
    for _ in range(MAXLEN + 2):
        for s in S:
            if is_terminal(s):
                continue
            V[s] = max(step(s, a)[1] + (0.0 if step(s, a)[2] else V[step(s, a)[0]])
                       for a in range(NA))
    return V

V_STAR = value_iteration_tokenworld()
Q_STAR = {(s, a): (lambda o: o[1] + (0.0 if o[2] else V_STAR[o[0]]))(step(s, a))
          for s in NONTERM for a in range(NA)}
print(f"scoreboard carried from Day 1:  V*(empty) = {V_STAR[()]:.4f}")

## 2. Monte Carlo against temporal difference learning

*Lecture: [Learning without a model](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#learning-without-a-model)*

The same behaviour policy, the same data and ten seeds; the errors are measured against the exact value of the policy.

**From the formulas to the code.** `behaviour_policy(eps=0.35)` returns the probabilities $\pi(a \mid s)$ of the policy that is evaluated: $0.35 / 4 = 0.0875$ for every token and $0.65$ more for the best one. The loop under `# ground truth` applies the Bellman equation of the policy until `V_pi` holds the exact values $V^{\pi}(s)$. In `evaluate_mc_td`, the line `G = r + G` builds the return $G_t$ backwards from the end of the episode, with $\gamma = 1$, and `V_mc[s] += alpha * (G - V_mc[s])` is the Monte Carlo update. In the second loop, `target` is the target $r_{t+1} + \gamma \hat{V}(s_{t+1})$ of temporal difference (TD) learning, with zero for a finished sentence, and `target - V_td[s]` is the TD error $\delta_t$. The code makes the TD updates after the episode, in the order of its steps, which gives the same numbers as updating after every step, because no prefix occurs twice in an episode. `rmse` is the root mean square error (RMSE) over the 40 prefixes. In the printed lines, MC stands for Monte Carlo and `+/-` for the standard deviation over the ten seeds, and the figure labels the one-step TD method TD(0).

In [ ]:
def behaviour_policy(eps=0.35):
    """A fixed stochastic policy whose value is estimated. Both methods must agree on it."""
    def pol(s):
        a_star = int(np.argmax([Q_STAR[(s, a)] for a in range(NA)]))
        p = np.full(NA, eps / NA); p[a_star] += 1 - eps
        return p
    return pol

POL = behaviour_policy()

# ground truth: exact policy evaluation of POL
V_pi = {s: 0.0 for s in S}
for _ in range(MAXLEN + 2):
    for s in S:
        if is_terminal(s):
            continue
        p = POL(s)
        V_pi[s] = sum(p[a] * (step(s, a)[1] + (0.0 if step(s, a)[2] else V_pi[step(s, a)[0]]))
                      for a in range(NA))
print(f"exact value of the behaviour policy: V_pi(empty) = {V_pi[()]:.4f}")

def evaluate_mc_td(n_episodes=3000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    V_mc = {s: 0.0 for s in S}; V_td = {s: 0.0 for s in S}
    err_mc, err_td = [], []
    rmse = lambda V: np.sqrt(np.mean([(V[s] - V_pi[s]) ** 2 for s in NONTERM]))
    for ep in range(n_episodes):
        traj, _ = rollout(POL, rng)
        G = 0.0
        for (s, a, r) in reversed(traj):                     # Monte Carlo
            G = r + G
            V_mc[s] += alpha * (G - V_mc[s])
        for i, (s, a, r) in enumerate(traj):                 # one-step TD
            s2 = s + (a,)
            target = r + (0.0 if is_terminal(s2) else V_td[s2])
            V_td[s] += alpha * (target - V_td[s])
        err_mc.append(rmse(V_mc)); err_td.append(rmse(V_td))
    return np.array(err_mc), np.array(err_td)

runs = [evaluate_mc_td(seed=s) for s in range(10)]
MC = np.array([r[0] for r in runs]); TD = np.array([r[1] for r in runs])

fig, ax = plt.subplots(figsize=(7, 3.4))
for arr, nm, col in ((MC, "Monte Carlo", "#c0392b"), (TD, "TD(0)", "#2980b9")):
    m = arr.mean(0)
    ax.plot(m, color=col, label=f"{nm}  (final RMSE {m[-1]:.4f})")
    ax.fill_between(np.arange(arr.shape[1]), np.percentile(arr, 10, 0),
                    np.percentile(arr, 90, 0), color=col, alpha=.15)
ax.set_xlabel("episodes"); ax.set_ylabel("RMSE against the exact V_pi")
ax.set_yscale("log"); ax.legend(fontsize=8)
ax.set_title("Same policy, same data, two estimators (10 seeds)")
plt.tight_layout(); plt.show()

for k in (100, 500, 3000):
    print(f"after {k:5d} episodes:  MC {MC[:, k-1].mean():.4f} +/- {MC[:, k-1].std():.4f}   "
          f"TD {TD[:, k-1].mean():.4f} +/- {TD[:, k-1].std():.4f}")

**Exercise A.** Which estimator has the lower mean error after 3000 episodes? Store `"MC"` or `"TD"` in `better`.

In [ ]:
better = None   # your answer

check("Exercise A", better, "MC" if MC[:, -1].mean() < TD[:, -1].mean() else "TD")

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
better = "MC" if MC[:, -1].mean() < TD[:, -1].mean() else "TD"

check("Exercise A", better, "MC" if MC[:, -1].mean() < TD[:, -1].mean() else "TD")

## 3. SARSA and Q-learning on TokenWorld &#91;[2](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#ref-2), [3](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#ref-3)&#93;

*Lecture: [SARSA and Q-learning](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#sarsa-and-q-learning)*

SARSA is named after the five items that one of its updates uses: state, action, reward, next state and next action.

**From the formulas to the code.** `Q[(s, a)]` is the table entry $\hat{Q}(s, a)$, and the two expressions that choose `a` and `a2` are the $\varepsilon$-greedy rule, where `eps` is $\varepsilon = 0.2$. `boot` is the only difference between the methods: `Q[(s2, a2)]` for SARSA and `max(Q[(s2, x)] ...)` for Q-learning. The update `Q[(s, a)] += alpha * (r + boot - Q[(s, a)])` has no `gamma` because $\gamma = 1$, and when the episode ends the target is `r` alone. Both methods choose `a2` before the update and take it as their next action, and Q-learning leaves it out of its target. Among equal entries `np.argmax` returns the first one, so a new table prefers the tokens in the order `the`, `cat`, `sat`, `<eos>`. `greedy_return(Q)` follows the largest entry $\hat{Q}(s, a)$ from the empty sentence and returns the reward of the sentence that results. `q_rmse(Q)` is the RMSE of the 160 entries against $Q^*$, and an entry that is missing from the table counts as zero. These two functions give the two curves of the figure, measured every 20 episodes.

In [ ]:
print(f"{len(NONTERM)} non-terminal states x {NA} actions = {len(Q_STAR)} Q-values to learn")

def greedy_return(Q):
    """Deterministic return of acting greedily with respect to a learned Q table."""
    s = ()
    while not is_terminal(s):
        a = int(np.argmax([Q.get((s, a_), 0.0) for a_ in range(NA)]))
        s, r, done = step(s, a)
    return r

def q_rmse(Q):
    return float(np.sqrt(np.mean([(Q.get(k, 0.0) - v) ** 2 for k, v in Q_STAR.items()])))

def tabular_control(method="q-learning", n_episodes=4000, alpha=0.1, eps=0.2, seed=0):
    rng = np.random.default_rng(seed)
    Q = {(s, a): 0.0 for s in NONTERM for a in range(NA)}
    hist = []
    for ep in range(n_episodes):
        s = ()
        a = (int(rng.integers(NA)) if rng.random() < eps
             else int(np.argmax([Q[(s, x)] for x in range(NA)])))
        while True:
            s2, r, done = step(s, a)
            if done:
                Q[(s, a)] += alpha * (r - Q[(s, a)])
                break
            a2 = (int(rng.integers(NA)) if rng.random() < eps
                  else int(np.argmax([Q[(s2, x)] for x in range(NA)])))
            boot = (Q[(s2, a2)] if method == "sarsa"
                    else max(Q[(s2, x)] for x in range(NA)))
            Q[(s, a)] += alpha * (r + boot - Q[(s, a)])
            s, a = s2, a2
        if ep % 20 == 0:
            hist.append((ep, greedy_return(Q), q_rmse(Q)))
    return Q, pd.DataFrame(hist, columns=["episode", "greedy_return", "Q_rmse"])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
final = {}
for method, col in (("q-learning", "#2980b9"), ("sarsa", "#c0392b")):
    runs = [tabular_control(method, seed=s)[1] for s in range(10)]
    R = np.array([r.greedy_return.values for r in runs])
    E = np.array([r.Q_rmse.values for r in runs])
    x = runs[0].episode.values
    ax[0].plot(x, R.mean(0), color=col, label=method)
    ax[0].fill_between(x, np.percentile(R, 10, 0), np.percentile(R, 90, 0), color=col, alpha=.15)
    ax[1].plot(x, E.mean(0), color=col, label=method)
    final[method] = (R[:, -1].mean(), E[:, -1].mean())
ax[0].axhline(V_STAR[()], ls="--", color="#111111", lw=1, label="exact optimum 2.75")
ax[0].set_xlabel("episodes"); ax[0].set_ylabel("return of the greedy policy"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("episodes"); ax[1].set_ylabel("RMSE of Q against Q*"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for m, (r_, e_) in final.items():
    print(f"{m:11s} greedy return {r_:.3f} of {V_STAR[()]:.3f}  "
          f"({100*r_/V_STAR[()]:.1f}% of optimum)   Q-RMSE {e_:.3f}")

## 4. The cliff

*Lecture: [The cliff](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#the-cliff)*

**From the formulas to the code.** `cliff_control` is the function of section 3 on the grid: `pick` is the $\varepsilon$-greedy rule with $\varepsilon = 0.1$, `boot` separates SARSA from Q-learning, and the update uses $\alpha = 0.5$ and $\gamma = 1$. The call passes `slip=0.0`, so `grid_step` never replaces an action: The wind is switched off. `total` adds up the rewards of one episode, the online return, and `online_return_last100` is its mean over the last 100 episodes of all ten seeds. `greedy_path_len` and `min_row_visited` describe the greedy path of the first seed: its number of steps and the highest row that it reaches, where row 0 is the top row and row 3 the row of the cliff. The loop `for _ in range(300)` cuts an episode off after 300 steps.

In [ ]:
def cliff_control(method, n_episodes=800, alpha=0.5, eps=0.1, gamma=1.0, slip=0.0, seed=0):
    rng = np.random.default_rng(seed)
    Q = {((r, c), a): 0.0 for r in range(ROWS) for c in range(COLS) for a in range(4)}
    online = []
    pick = lambda s: (int(rng.integers(4)) if rng.random() < eps
                      else int(np.argmax([Q[(s, x)] for x in range(4)])))
    for ep in range(n_episodes):
        s = START; a = pick(s); total = 0.0
        for _ in range(300):
            s2, r, done = grid_step(s, a, slip=slip, rng=rng)
            total += r
            if done:
                Q[(s, a)] += alpha * (r - Q[(s, a)]); break
            a2 = pick(s2)
            boot = Q[(s2, a2)] if method == "sarsa" else max(Q[(s2, x)] for x in range(4))
            Q[(s, a)] += alpha * (r + gamma * boot - Q[(s, a)])
            s, a = s2, a2
        online.append(total)
    return Q, np.array(online)

rows = []
paths = {}
for method in ("q-learning", "sarsa"):
    onl = []
    for sd in range(10):
        Q, o = cliff_control(method, seed=sd)
        onl.append(o)
        if sd == 0:
            paths[method] = Q
    onl = np.array(onl)
    # greedy path length and safety
    Q = paths[method]; s = START; path = [s]
    for _ in range(60):
        a = int(np.argmax([Q[(s, x)] for x in range(4)]))
        s, r, done = grid_step(s, a, slip=0.0)
        path.append(s)
        if done:
            break
    rows.append(dict(method=method, online_return_last100=onl[:, -100:].mean(),
                     greedy_path_len=len(path) - 1,
                     min_row_visited=min(p[0] for p in path)))
    paths[method] = path
print(pd.DataFrame(rows).round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 2.6))
board = np.zeros((ROWS, COLS))
for cpos in CLIFF:
    board[cpos] = -1
ax.imshow(board, cmap="RdGy", vmin=-1, vmax=1)
for method, col, off in (("q-learning", "#2980b9", -0.15), ("sarsa", "#c0392b", 0.15)):
    p = np.array(paths[method])
    ax.plot(p[:, 1], p[:, 0] + off, "-o", ms=4, color=col, label=method)
ax.set_xticks([]); ax.set_yticks([]); ax.legend(fontsize=8, loc="upper right")
ax.set_title("Greedy paths after training (dark band = cliff)")
plt.tight_layout(); plt.show()

**Exercise B.** Store the return of SARSA during the last 100 learning episodes, `rows[1]["online_return_last100"]`, in `sarsa_online`.

In [ ]:
sarsa_online = None   # your answer

check("Exercise B", sarsa_online, float(rows[1]["online_return_last100"]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
sarsa_online = float(rows[1]["online_return_last100"])

check("Exercise B", sarsa_online, float(rows[1]["online_return_last100"]))

## 5. A deep Q-network in NumPy &#91;[4](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#ref-4), [5](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#ref-5)&#93;

*Lecture: [Deep Q-networks](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#deep-q-networks)*

**Python move: a class with a forward and a backward pass.** `QNet` stores the weights, computes the values in `forward` and applies one step of the Adam optimiser to the gradient of the Huber loss in `train_step`. Deep-learning libraries automate exactly these lines.

**From the formulas to the code.** `encode_state` builds the 20 numbers $x$ of the one-hot encoding. In `forward`, `np.maximum(0.0, X @ self.W1 + self.b1)` is the hidden layer $h$ with the rectified linear unit (ReLU) and `self.H @ self.W2 + self.b2` gives the four Q-values. The matrix stands on the right in `X @ W1` because the code stores one state per row of `X`. In `dqn`, the line that computes `Y` is the target $y$: `Dn` is 1 for a terminal next state, which removes the bootstrap term, and `tgt` is the target network with the weights $w^-$. `train_step` never computes the loss itself: `np.clip(..., -1.0, 1.0)` is the slope of the Huber loss with respect to the error, and the lines after it pass this slope back through the two layers to obtain the gradient $g$ of every weight. The loop marked `# Adam` is the Adam update, with `self.t` as the counter $k$ and `lr` as $\alpha$. `buf` is the replay buffer, `sample` the batch and `tgt.copy_from(net)` the copy made every `sync` steps. The printed table calls the complete deep Q-network (DQN) *full DQN*, and its numbers are those of the last check, at episode 675.

In [ ]:
def encode_state(s):
    """One-hot the prefix: MAXLEN positions x (vocabulary + 1 for an empty position)."""
    x = np.zeros(MAXLEN * (NA + 1))
    for i in range(MAXLEN):
        x[i * (NA + 1) + (s[i] if i < len(s) else NA)] = 1.0
    return x

class QNet:
    """Q(x) = W2 relu(W1 x + b1) + b2, trained with the Huber loss and the Adam optimiser."""
    def __init__(self, d_in, d_hidden, d_out, rng, lr=3e-3):
        self.W1 = rng.normal(0, 1 / np.sqrt(d_in), (d_in, d_hidden)); self.b1 = np.zeros(d_hidden)
        self.W2 = rng.normal(0, 1 / np.sqrt(d_hidden), (d_hidden, d_out)); self.b2 = np.zeros(d_out)
        self.lr, self.t = lr, 0
        self.m = [np.zeros_like(p) for p in self.params()]
        self.v = [np.zeros_like(p) for p in self.params()]

    def params(self):
        return [self.W1, self.b1, self.W2, self.b2]

    def forward(self, X):
        self.X = X
        self.H = np.maximum(0.0, X @ self.W1 + self.b1)
        return self.H @ self.W2 + self.b2

    def copy_from(self, other):
        for p, q in zip(self.params(), other.params()):
            p[...] = q

    def train_step(self, X, A, Y):
        Q = self.forward(X); idx = np.arange(len(A))
        gQ = np.zeros_like(Q); gQ[idx, A] = np.clip(Q[idx, A] - Y, -1.0, 1.0) / len(A)   # Huber gradient
        gW2, gb2 = self.H.T @ gQ, gQ.sum(0)
        gH = gQ @ self.W2.T; gH[self.H <= 0] = 0.0
        gW1, gb1 = self.X.T @ gH, gH.sum(0)
        self.t += 1
        for i, (p, g) in enumerate(zip(self.params(), (gW1, gb1, gW2, gb2))):          # Adam
            self.m[i] = 0.9 * self.m[i] + 0.1 * g
            self.v[i] = 0.999 * self.v[i] + 0.001 * g * g
            p -= self.lr * (self.m[i] / (1 - 0.9 ** self.t)) / (np.sqrt(self.v[i] / (1 - 0.999 ** self.t)) + 1e-8)

def dqn(n_episodes=700, replay=True, target_net=True, lr=3e-3, batch=64, buffer_cap=4000, sync=100, eps0=0.6, seed=0):
    rng = np.random.default_rng(seed); D = MAXLEN * (NA + 1)
    net, tgt = QNet(D, 64, NA, rng, lr), QNet(D, 64, NA, rng, lr)
    tgt.copy_from(net)
    buf, hist, gstep = [], [], 0
    X_all = np.stack([encode_state(s) for s in NONTERM])

    def q_table():
        out = net.forward(X_all)
        return {(s, a): float(out[i, a]) for i, s in enumerate(NONTERM) for a in range(NA)}

    for ep in range(n_episodes):
        eps = max(0.05, eps0 * (1 - ep / n_episodes))
        s = ()
        while not is_terminal(s):
            x = encode_state(s)
            a = int(rng.integers(NA)) if rng.random() < eps else int(net.forward(x[None])[0].argmax())
            s2, r, done = step(s, a)
            buf.append((x, a, r, encode_state(s2), float(done)))
            if len(buf) > buffer_cap:
                buf.pop(0)
            s = s2; gstep += 1
            if replay:
                if len(buf) < batch:
                    continue
                sample = [buf[i] for i in rng.integers(0, len(buf), batch)]
            else:
                sample = [buf[-1]]                              # the ablation: the last transition only
            X = np.stack([b[0] for b in sample]); A = np.array([b[1] for b in sample])
            R = np.array([b[2] for b in sample]); X2 = np.stack([b[3] for b in sample])
            Dn = np.array([b[4] for b in sample])
            Y = R + (1 - Dn) * (tgt if target_net else net).forward(X2).max(1)
            net.train_step(X, A, Y)
            if target_net and gstep % sync == 0:
                tgt.copy_from(net)
        if ep % 25 == 0:
            Q = q_table(); hist.append((ep, greedy_return(Q), q_rmse(Q)))
    Q = q_table()
    return pd.DataFrame(hist, columns=["episode", "greedy_return", "Q_rmse"]), Q

import time
configs = {"full DQN": dict(replay=True, target_net=True), "no replay": dict(replay=False, target_net=True),
           "no target net": dict(replay=True, target_net=False), "neither": dict(replay=False, target_net=False)}
res, curves = [], {}
for nm, kw in configs.items():
    t0 = time.time()
    runs = [dqn(seed=s, **kw) for s in range(3)]
    R = np.array([r[0].greedy_return.values for r in runs]); E = np.array([r[0].Q_rmse.values for r in runs])
    curves[nm] = (runs[0][0].episode.values, R, E)
    res.append(dict(config=nm, final_return=R[:, -1].mean(), pct_of_optimum=100 * R[:, -1].mean() / V_STAR[()],
                    Q_rmse=E[:, -1].mean(), seconds=(time.time() - t0) / 3))
dqn_table = pd.DataFrame(res)
print(dqn_table.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
for nm, col in (("full DQN", "#27ae60"), ("no replay", "#e67e22"),
                ("no target net", "#2980b9"), ("neither", "#c0392b")):
    x, R, E = curves[nm]
    ax[0].plot(x, R.mean(0), color=col, label=nm)
    ax[1].plot(x, E.mean(0), color=col, label=nm)
ax[0].axhline(V_STAR[()], ls="--", lw=1, color="#111111", label="exact optimum")
ax[0].set_xlabel("episodes"); ax[0].set_ylabel("greedy return"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("episodes"); ax[1].set_ylabel("Q RMSE against Q*"); ax[1].set_yscale("log")
ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

best = max(res, key=lambda r: -r["Q_rmse"])
print(f"lowest Q-error: {best['config']}")

**Exercise C.** Store the error of the values of the full DQN from the table `dqn_table` in `full_err`.

In [ ]:
full_err = None   # your answer

check("Exercise C", full_err, float(dqn_table.set_index("config").loc["full DQN", "Q_rmse"]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
full_err = float(dqn_table.set_index("config").loc["full DQN", "Q_rmse"])

check("Exercise C", full_err, float(dqn_table.set_index("config").loc["full DQN", "Q_rmse"]))

## 6. Your field: a gridworld

*Lecture: [Your field](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#your-field)*

Change `DOMAIN` to `drone` or `evacuation` and run the cell again.

**From the formulas to the code.** `td_control` holds the same two updates as sections 3 and 4, with the table stored as a NumPy array `Q[row, column, action]`. `nxt` is the bootstrap term: `Q[s2].max()` for Q-learning, `Q[s2 + (a2,)]` for SARSA, and 0 when the episode has ended. The update uses $\alpha = 0.2$, $\gamma = 0.98$ and $\varepsilon = 0.1$. In `domain_step`, a slip replaces the action by a random one of the four with the probability `slip` of the domain, a wall or the border leaves the agent in its cell, and a hazard returns the penalty and the start cell. `td_table` holds the mean online return of the last 200 episodes over five seeds and the worst of these episodes.

In [ ]:
DOMAIN = "warehouse"     # choose: warehouse, drone or evacuation

def make_domain(name):
    """A grid with walls, hazards, a start, a goal and a chance that the agent slips in a random direction."""
    if name == "warehouse":     # shelves are walls; a forklift lane is dangerous
        return dict(rows=6, cols=8, walls={(1, 2), (2, 2), (3, 2), (1, 5), (2, 5), (3, 5), (4, 5)}, hazards={(5, 3), (5, 4), (4, 3)},
                    start=(5, 0), goal=(0, 7), slip=0.05, penalty=-20.0)
    if name == "drone":         # wind and a no-fly zone near the ground
        return dict(rows=5, cols=8, walls={(0, 3), (1, 3)}, hazards={(4, c) for c in range(1, 7)},
                    start=(3, 0), goal=(3, 7), slip=0.20, penalty=-50.0)
    if name == "evacuation":    # walls, fire and smoke
        return dict(rows=6, cols=6, walls={(2, 1), (2, 2), (2, 3), (4, 3), (4, 4)}, hazards={(3, 4), (1, 4), (3, 1)},
                    start=(5, 0), goal=(0, 5), slip=0.10, penalty=-100.0)
    raise ValueError(name)

MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def domain_step(D, s, a, rng):
    """One move; a slip replaces the action by a random one, walls block, a hazard costs its penalty and resets."""
    if rng.random() < D["slip"]:
        a = int(rng.integers(4))
    r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
    if not (0 <= r < D["rows"] and 0 <= c < D["cols"]) or (r, c) in D["walls"]:
        r, c = s
    if (r, c) in D["hazards"]:
        return D["start"], D["penalty"], False
    if (r, c) == D["goal"]:
        return (r, c), 0.0, True
    return (r, c), -1.0, False

def greedy_path(D, table, n=40):
    """The path that follows the best action from the start, without slips."""
    s, path = D["start"], [D["start"]]
    for _ in range(n):
        a = int(np.argmax(table[s])); r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
        if not (0 <= r < D["rows"] and 0 <= c < D["cols"]) or (r, c) in D["walls"]:
            break
        s = (r, c); path.append(s)
        if s == D["goal"] or s in D["hazards"]:
            break
    return path

def draw_domain(D, ax, paths=(), title=""):
    img = np.ones((D["rows"], D["cols"], 3))
    for w in D["walls"]: img[w] = (0.45, 0.45, 0.45)
    for h_ in D["hazards"]: img[h_] = (0.95, 0.55, 0.5)
    img[D["goal"]] = (0.55, 0.85, 0.55); img[D["start"]] = (0.6, 0.75, 0.95)
    ax.imshow(img)
    for p, col in paths:
        ax.plot([c for r, c in p], [r for r, c in p], "-o", color=col, ms=3, lw=2)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=9)


def td_control(D, method, episodes=800, alpha=0.2, gamma=0.98, eps=0.1, seed=0, max_steps=200):
    """Tabular SARSA or Q-learning with an epsilon-greedy behaviour policy."""
    rng = np.random.default_rng(seed); Q = np.zeros((D["rows"], D["cols"], 4)); returns = []
    pick = lambda s: int(rng.integers(4)) if rng.random() < eps else int(np.argmax(Q[s]))
    for _ in range(episodes):
        s = D["start"]; a = pick(s); G = 0.0
        for _ in range(max_steps):
            s2, r, done = domain_step(D, s, a, rng); a2 = pick(s2); G += r
            nxt = 0.0 if done else (Q[s2].max() if method == "Q-learning" else Q[s2 + (a2,)])
            Q[s + (a,)] += alpha * (r + gamma * nxt - Q[s + (a,)]); s, a = s2, a2
            if done:
                break
        returns.append(G)
    return Q, np.array(returns)

dom = make_domain(DOMAIN); td_runs = {}
for method in ("SARSA", "Q-learning"):
    runs = [td_control(dom, method, seed=k) for k in range(5)]
    td_runs[method] = (runs[0][0], np.array([r for _, r in runs]))
td_table = pd.DataFrame({m: [R[:, -200:].mean(), R[:, -200:].min()] for m, (_, R) in td_runs.items()},
                        index=["mean return while learning, last 200 episodes", "worst episode"]).T
print(f"{DOMAIN}:"); print(td_table.round(1).to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
for (m, (_, R)), col in zip(td_runs.items(), ("#2980b9", "#c0392b")):
    ax[0].plot(pd.Series(R.mean(0)).rolling(25).mean(), color=col, label=m)
ax[0].set_xlabel("episode"); ax[0].set_ylabel("return (moving average)"); ax[0].legend(fontsize=8)
draw_domain(dom, ax[1], [(greedy_path(dom, td_runs["SARSA"][0]), "#2980b9"), (greedy_path(dom, td_runs["Q-learning"][0]), "#c0392b")],
            "greedy paths: SARSA (blue), Q-learning (red)")
plt.tight_layout(); plt.show()

**Exercise D.** Which method collects more reward while learning in your domain? Store its name, `"SARSA"` or `"Q-learning"`, in `safer`.

In [ ]:
safer = None   # your answer

check("Exercise D", safer, td_table.iloc[:, 0].idxmax())

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
safer = td_table.iloc[:, 0].idxmax()

check("Exercise D", safer, td_table.iloc[:, 0].idxmax())

## 7. Going further (optional): schedules and the deadly triad

*Lecture: [Going further (optional)](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#going-further-optional)*

**From the formulas to the code.** In `schedules`, the dictionary after `eps =` holds the three exploration schedules and the dictionary after `alpha =` the three step-size schedules. `ep` is the episode number $i$ and `counts[(s, a)]` the number of updates $N(s, a)$, and the update is that of Q-learning with $\gamma = 1$. In `baird`, `X[s]` is the feature list $x(s)$, `X[s] @ w` the value $\hat{V}_w(s)$ and `rho` the importance ratio $\rho$. The line `w = w + alpha * rho * (boot - X[s] @ w) * X[s]` is the update of the weights, with a reward of zero. The code counts from 0, so the seventh state of the lecture is `s = 6` and the weight $w_8$ is `w[7]`. The options `features="tabular"`, `target="mc"` and `sampling="on-policy"` each remove one leg of the triad. `np.linalg.norm(w)` is the norm $\lVert w \rVert$, and the value printed as *start* is the norm after the first update.

### Exploration and step-size schedules

In [ ]:
def schedules(n_episodes=4000, seed=0, eps_mode="const", alpha_mode="const"):
    rng = np.random.default_rng(seed)
    Q = {(s, a): 0.0 for s in NONTERM for a in range(NA)}
    counts = {}
    for ep in range(n_episodes):
        eps = {"const": 0.2, "decay": max(0.02, 0.5 * (1 - ep / n_episodes)),
               "1/t": 1.0 / (1 + ep / 200)}[eps_mode]
        s = ()
        while not is_terminal(s):
            a = (int(rng.integers(NA)) if rng.random() < eps
                 else int(np.argmax([Q[(s, x)] for x in range(NA)])))
            counts[(s, a)] = counts.get((s, a), 0) + 1
            alpha = {"const": 0.1, "1/n": 1.0 / counts[(s, a)],
                     "poly": counts[(s, a)] ** -0.7}[alpha_mode]
            s2, r, done = step(s, a)
            boot = 0.0 if done else max(Q[(s2, x)] for x in range(NA))
            Q[(s, a)] += alpha * (r + boot - Q[(s, a)])
            s = s2
    return greedy_return(Q), q_rmse(Q)

rows = []
for em in ("const", "decay", "1/t"):
    for am in ("const", "1/n", "poly"):
        r = np.array([schedules(seed=s, eps_mode=em, alpha_mode=am) for s in range(8)])
        rows.append(dict(epsilon=em, alpha=am, greedy_return=r[:, 0].mean(),
                         optimal_seeds=f"{int((r[:, 0] >= V_STAR[()] - 1e-9).sum())}/8",
                         Q_rmse=r[:, 1].mean()))
sch = pd.DataFrame(rows).sort_values("Q_rmse")
print(sch.round(4).to_string(index=False))

### The deadly triad &#91;[1](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lecture.html#ref-1)&#93;

In [ ]:
def baird(n_steps=3000, alpha=0.01, gamma=0.99, features="linear", target="td", sampling="off-policy", seed=0):
    """Reward is zero everywhere, so the true value function is zero.

    features: "linear" uses Baird's eight features for seven states, "tabular" one feature per state.
    target:   "td" bootstraps from the estimate of the next state, "mc" uses the true return, which is zero.
    sampling: "off-policy" samples states and actions from the behaviour policy and corrects with importance
              ratios; "on-policy" follows the target policy, which always moves to state 6, from random restarts."""
    rng = np.random.default_rng(seed)
    if features == "linear":
        X = np.zeros((7, 8))
        for i in range(6):
            X[i, i] = 2.0; X[i, 7] = 1.0
        X[6, 6] = 1.0; X[6, 7] = 2.0
        w = np.ones(8); w[6] = 10.0
    else:
        X = np.eye(7); w = np.array([3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 12.0])   # the same initial values as the linear case
    norms, s = [], int(rng.integers(7))
    for t in range(n_steps):
        if sampling == "off-policy":
            s = int(rng.integers(7))
            if rng.random() < 6 / 7:
                s2, rho = int(rng.integers(6)), 0.0          # dashed action: the target policy never takes it
            else:
                s2, rho = 6, 7.0                             # solid action: importance ratio 1 / (1/7)
        else:
            s2, rho = 6, 1.0
        boot = gamma * X[s2] @ w if target == "td" else 0.0
        w = w + alpha * rho * (boot - X[s] @ w) * X[s]
        if sampling == "on-policy":
            s = s2 if rng.random() > 0.1 else int(rng.integers(7))
        norms.append(float(np.linalg.norm(w)))
    return np.array(norms)

variants = {"all three legs": dict(),
            "tabular features": dict(features="tabular"),
            "Monte Carlo targets": dict(target="mc"),
            "on-policy sampling": dict(sampling="on-policy")}
fig, ax = plt.subplots(figsize=(8, 3.4))
triad = {}
for name, kw in variants.items():
    n = baird(**kw); triad[name] = n
    ax.semilogy(n, label=f"{name}: {n[-1]:.3g}")
ax.set_xlabel("updates"); ax.set_ylabel("norm of the weights (log scale)"); ax.legend(fontsize=8)
ax.set_title("Baird's counterexample: remove any one leg of the triad"); plt.show()
for name, n in triad.items():
    print(f"{name:20s} norm after {len(n)} updates: {n[-1]:10.3g}   (start {n[0]:.2f})")

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Add Double Q-learning to the deep Q-network (DQN) of section 5 and measure whether it changes the error of the values on TokenWorld. |
| Electronics and Communication Engineering | Formulate the choice of a modulation scheme under a changing channel as a small Markov decision process (MDP) and learn it with Q-learning. |
| Electrical and Electronics Engineering | Use SARSA to learn a charging policy for a battery with three states of charge and a two-level electricity price. |
| Mechanical Engineering | Learn a maintenance policy for a machine with four wear levels with Q-learning and compare it with the policy from value iteration. |
| Biomedical Engineering | Explain, with the cliff example, why an on-policy method might be preferred for a treatment policy that keeps exploring in practice. |
| Civil Engineering | Learn a traffic-light policy for one intersection with two queue lengths per direction and report the spread over ten seeds. |
| Aeronautical Engineering | Repeat the deadly-triad experiment with a smaller step size and report whether the divergence disappears or only slows down. |
| Biotechnology | Model a feeding schedule for a bioreactor as an MDP with three biomass levels and learn it with SARSA. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-02/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-02).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>